In [9]:
# import torch
# import torch.nn as nn

# class LanguageEmbedding(nn.Module):
#     def __init__(self, embedding_dim):
#         super(LanguageEmbedding, self).__init__()
#         self.embedding = nn.Linear(embedding_dim, embedding_dim)

#     def forward(self, sentence_embeddings):
#         return self.embedding(sentence_embeddings)
    
# class MeaningEmbedding(nn.Module):
#     def __init__(self, embedding_dim):
#         super(MeaningEmbedding, self).__init__()
#         self.embedding = nn.Linear(embedding_dim, embedding_dim)

#     def forward(self, sentence_embeddings):
#         return self.embedding(sentence_embeddings)
    
# class LanguageIdentifier(nn.Module):
#     def __init__(self, embedding_dim, num_languages=8):
#         super(LanguageIdentifier, self).__init__()
#         self.classifier = nn.Linear(embedding_dim, num_languages)

#     def forward(self, language_embedding):
#         return self.classifier(language_embedding)
    
# class DREAMModel(nn.Module):
#     def __init__(self, embedding_dim):
#         super(DREAMModel, self).__init__()
#         self.language_embedding = LanguageEmbedding(embedding_dim)
#         self.meaning_embedding = MeaningEmbedding(embedding_dim)
#         self.language_identifier = LanguageIdentifier(embedding_dim)

#     def forward(self, sentence_embeddings):
#         language_embedded = self.language_embedding(sentence_embeddings)
#         meaning_embedded = self.meaning_embedding(sentence_embeddings)
#         language_id = self.language_identifier(language_embedded)
#         return language_embedded, meaning_embedded, language_id

In [10]:
import torch
import torch.nn as nn

class LanguageEmbedding(nn.Module):
    def __init__(self, embedding_dim):
        super(LanguageEmbedding, self).__init__()
        self.embedding = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, sentence_embeddings):
        return self.embedding(sentence_embeddings)
    
class MeaningEmbedding(nn.Module):
    def __init__(self, embedding_dim):
        super(MeaningEmbedding, self).__init__()
        self.embedding = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, sentence_embeddings):
        return self.embedding(sentence_embeddings)
    
class LanguageIdentifier(nn.Module):
    def __init__(self, embedding_dim, num_languages=8):
        super(LanguageIdentifier, self).__init__()
        self.classifier = nn.Linear(embedding_dim, num_languages)

    def forward(self, language_embedding):
        return self.classifier(language_embedding)
    
class DREAMModel(nn.Module):
    def __init__(self, embedding_dim, num_languages=8):
        super(DREAMModel, self).__init__()
        self.language_embedding = LanguageEmbedding(embedding_dim)
        self.meaning_embedding = MeaningEmbedding(embedding_dim)
        self.language_identifier = LanguageIdentifier(embedding_dim, num_languages)

    def forward(self, sentence_embeddings):
        language_embedded = self.language_embedding(sentence_embeddings)
        meaning_embedded = self.meaning_embedding(sentence_embeddings)
        language_id = self.language_identifier(language_embedded)
        return language_embedded, meaning_embedded, language_id

In [11]:
# import torch
# import pandas as pd
# from pathlib import Path
# from transformers import AutoTokenizer, AutoModel
# from torch.utils.data import Dataset, DataLoader
# from typing import List


# class EmbeddingEncoder:
#     """
#     Chạy XLM-R trên raw text, lưu embeddings ra file .pt.
#     Chỉ chạy 1 lần — kết quả dùng lại cho mọi training run.
#     """

#     def __init__(self, model_name: str = "xlm-roberta-large"):
#         self.device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         self.tokenizer = AutoTokenizer.from_pretrained(model_name)
#         self.model     = AutoModel.from_pretrained(model_name).to(self.device)
#         self.model.eval()   # freeze — không update weights
#         print(f"Loaded {model_name} on {self.device}")

#     @torch.no_grad()    # không build graph — tiết kiệm memory
#     def encode(self, sentences: List[str], batch_size: int = 64) -> torch.Tensor:
#         """
#         Encode list of strings → tensor (N, d).
#         Dùng [CLS] token theo đúng paper Section 3.4.
#         """
#         all_embeddings = []

#         for i in range(0, len(sentences), batch_size):
#             batch = sentences[i : i + batch_size]

#             inputs = self.tokenizer(
#                 batch,
#                 padding        = True,
#                 truncation     = True,
#                 max_length     = 128,
#                 return_tensors = "pt",
#             ).to(self.device)

#             outputs = self.model(**inputs)

#             # outputs.last_hidden_state: (B, seq_len, hidden_dim)
#             # [:, 0, :] → [CLS] token của mỗi câu → (B, hidden_dim)
#             cls_embeddings = outputs.last_hidden_state[:, 0, :]

#             all_embeddings.append(cls_embeddings.cpu())  # về CPU để lưu

#             if (i // batch_size) % 10 == 0:
#                 print(f"  {i}/{len(sentences)}")

#         return torch.cat(all_embeddings, dim=0)   # (N, d)

#     def encode_and_save(self, dataset: "TatoebaDataset", save_path: str):
#         """
#         Encode toàn bộ dataset và lưu ra file .pt.

#         File được lưu chứa:
#             src_embeddings : (N, d)
#             trg_embeddings : (N, d)
#             src_lang_ids   : (N,)
#             trg_lang_ids   : (N,)
#         """
#         save_path = Path(save_path)
#         save_path.parent.mkdir(parents=True, exist_ok=True)

#         print(f"Encoding {len(dataset):,} sentence pairs...")

#         print("Encoding sources...")
#         src_embeddings = self.encode(dataset.sources)

#         print("Encoding targets...")
#         trg_embeddings = self.encode(dataset.targets)

#         torch.save({
#             "src_embeddings" : src_embeddings,
#             "trg_embeddings" : trg_embeddings,
#             "src_lang_ids"   : torch.zeros(len(dataset), dtype=torch.long),  # English = 0
#             "trg_lang_ids"   : torch.tensor(dataset.target_lang_ids, dtype=torch.long),
#         }, save_path)

#         print(f"Saved → {save_path}")
#         print(f"  src_embeddings : {src_embeddings.shape}")
#         print(f"  trg_embeddings : {trg_embeddings.shape}")

In [12]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

class EmbeddingEncoder:
    def __init__(self, model_name: str = "sentence-transformers/LaBSE"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model  = SentenceTransformer(model_name, device=str(self.device))
        self.model.eval()
        print(f"Loaded {model_name} on {self.device}")

    @torch.no_grad()
    def encode(self, sentences: list, batch_size: int = 64) -> torch.Tensor:
        all_embeddings = []
        batches = range(0, len(sentences), batch_size)

        for i in tqdm(batches, desc="Encoding", unit="batch",
                      total=len(batches), dynamic_ncols=True):
            batch = sentences[i : i + batch_size]

            embeddings = self.model.encode(
                batch,
                batch_size        = batch_size,
                convert_to_tensor = True,
                show_progress_bar = False,   # tqdm của mình handle rồi
                device            = str(self.device),
            )
            all_embeddings.append(embeddings.cpu())

        return torch.cat(all_embeddings, dim=0)   # (N, 768)

    def encode_and_save(self, dataset, save_path: str):
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        print(f"\nEncoding {len(dataset):,} sentence pairs...")

        print("Sources:")
        src_embeddings = self.encode(dataset.sources)

        print("Targets:")
        trg_embeddings = self.encode(dataset.targets)

        torch.save({
            "src_embeddings" : src_embeddings,
            "trg_embeddings" : trg_embeddings,
            "src_lang_ids"   : torch.zeros(len(dataset), dtype=torch.long),
            "trg_lang_ids"   : torch.tensor(dataset.target_lang_ids, dtype=torch.long),
        }, save_path)

        print(f"\nSaved → {save_path}")
        print(f"  src_embeddings : {src_embeddings.shape}")
        print(f"  trg_embeddings : {trg_embeddings.shape}")

In [13]:
# import glob
# import torch
# import torch.nn.functional as F
# import pandas as pd
# from pathlib import Path
# from scipy.stats import pearsonr


# def evaluate_sts17(
#     model          : DREAMModel,
#     sts17_dir      : str,
#     encoder        : EmbeddingEncoder,
#     batch_size     : int = 256,
# ) -> dict:
#     """
#     Tính Pearson correlation trên từng language pair của STS-17.

#     Format file: <src>\t<trg>\t<score>  (không có header)
#     Score: 0–5, càng cao càng giống nghĩa.

#     Returns:
#         dict: { "en-de": 0.72, "en-fr": 0.68, ..., "mean": 0.70 }
#     """
#     model.eval()
#     device = next(model.parameters()).device

#     tsv_files = sorted(glob.glob(f"{sts17_dir}/*.txt"))
#     if not tsv_files:
#         raise FileNotFoundError(f"Không tìm thấy .txt nào trong {sts17_dir}")

#     results = {}

#     for file_path in tsv_files:
#         lang_pair = Path(file_path).stem   # "STS.gs.en-de" → dùng làm key
#         # Lấy "en-de" từ tên file nếu có dấu "."
#         if "." in lang_pair:
#             lang_pair = lang_pair.split(".")[-1]

#         df = pd.read_csv(
#             file_path,
#             sep      = "\t",
#             header   = None,
#             names    = ["src", "trg", "score"],
#             on_bad_lines = "skip",
#         ).dropna()
#         print(df.shape)

#         if len(df) == 0:
#             print(f"⚠️  {lang_pair}: rỗng, bỏ qua.")
#             continue

#         # Encode cả 2 chiều
#         with torch.no_grad():
#             src_emb = encoder.encode(df["src"].tolist(), batch_size).to(device)
#             trg_emb = encoder.encode(df["trg"].tolist(), batch_size).to(device)

#             # Lấy meaning embedding — đây là representation dùng để tính similarity
#             _, src_eM, _ = model(src_emb)
#             _, trg_eM, _ = model(trg_emb)

#             cosine_scores = F.cosine_similarity(src_eM, trg_eM, dim=-1).cpu().numpy()

#         gold_scores = df["score"].astype(float).values

#         pearson, _ = pearsonr(cosine_scores, gold_scores)
#         results[lang_pair] = pearson

#         print(f"  {lang_pair:10s} | n={len(df):4d} | Pearson={pearson:.4f}")

#     # Macro average — mỗi language pair đóng góp ngang nhau
#     results["mean"] = sum(results.values()) / len(results)
#     print(f"\n  {'mean':10s} |       | Pearson={results['mean']:.4f}")

#     return results

In [14]:
import glob
import torch
import torch.nn.functional as F
import pandas as pd
from pathlib import Path
from scipy.stats import pearsonr


def evaluate_sts17_full(
    model      : DREAMModel,
    sts17_dir  : str,
    encoder    : EmbeddingEncoder,
    batch_size : int = 256,
) -> pd.DataFrame:
    """
    Chạy cả 2 rows giống paper:
        XLM-R          → raw CLS embedding
        XLM-R (Meaning)→ sau khi qua DREAM meaning_embedding
    """
    model.eval()
    device = next(model.parameters()).device

    txt_files = sorted(glob.glob(f"{sts17_dir}/*.txt"))
    if not txt_files:
        raise FileNotFoundError(f"Không tìm thấy .txt nào trong {sts17_dir}")

    # lang_pair → {"raw": pearson, "meaning": pearson}
    scores = {}

    for file_path in txt_files:
        # Parse tên file → lang pair key
        stem = Path(file_path).stem
        lang_pair = stem.split(".")[-1] if "." in stem else stem

        df = pd.read_csv(
            file_path,
            sep          = "\t",
            header       = None,
            names        = ["src", "trg", "score"],
            on_bad_lines = "skip",
        ).dropna()

        if len(df) == 0:
            continue

        gold = df["score"].astype(float).values

        with torch.no_grad():
            src_emb = encoder.encode(df["src"].tolist(), batch_size).to(device)
            trg_emb = encoder.encode(df["trg"].tolist(), batch_size).to(device)

            # ── Baseline: raw XLM-R CLS ──────────────────────
            raw_sim = F.cosine_similarity(src_emb, trg_emb, dim=-1).cpu().numpy()
            raw_pearson, _ = pearsonr(raw_sim, gold)

            # ── Meaning: qua DREAM ───────────────────────────
            _, src_eM, _ = model(src_emb)
            _, trg_eM, _ = model(trg_emb)
            meaning_sim = F.cosine_similarity(src_eM, trg_eM, dim=-1).cpu().numpy()
            meaning_pearson, _ = pearsonr(meaning_sim, gold)

        scores[lang_pair] = {"XLM-R": raw_pearson, "XLM-R (Meaning)": meaning_pearson}
        print(f"  {lang_pair:12s} | XLM-R={raw_pearson:.3f} | Meaning={meaning_pearson:.3f}")

    # ── Build DataFrame giống paper ──────────────────────────
    df_results = pd.DataFrame(scores).T   # rows=lang_pair, cols=model
    df_results.index.name = "lang_pair"

    # Thêm cột Avg
    df_results["Avg"] = df_results.mean(axis=1)

    # Transpose để rows=model, cols=lang_pair (giống paper)
    df_results = df_results.T

    # Bold max per column khi print
    print("\n")
    print(df_results.round(3).to_string())

    return df_results


# ── Chạy ────────────────────────────────────────────────────
model = DREAMModel(embedding_dim=768)
model.load_state_dict(torch.load("checkpoints_labse/best.pt", weights_only=True))
model = model.to("cuda")

# Dùng lại encoder đã có
#encoder = EmbeddingEncoder("xlm-roberta-large")
encoder = EmbeddingEncoder("sentence-transformers/LaBSE")

df_results = evaluate_sts17_full(
    model      = model,
    sts17_dir  = r"..\data\STS17",
    encoder    = encoder,   # dùng lại encoder đã load
)

Loaded sentence-transformers/LaBSE on cuda


Encoding: 100%|██████████| 1/1 [00:00<00:00, 12.41batch/s]


  en-ar        | XLM-R=0.722 | Meaning=0.695


Encoding: 100%|██████████| 1/1 [00:00<00:00, 15.18batch/s]


  en-de        | XLM-R=0.741 | Meaning=0.715


...

  fr-en        | XLM-R=0.763 | Meaning=0.746


Encoding: 100%|██████████| 1/1 [00:00<00:00, 14.11batch/s]


  it-en        | XLM-R=0.765 | Meaning=0.748


Encoding: 100%|██████████| 1/1 [00:00<00:00, 13.21batch/s]

  nl-en        | XLM-R=0.750 | Meaning=0.742


lang_pair        en-ar  en-de  en-tr  es-en  fr-en  it-en  nl-en
XLM-R            0.722  0.741  0.729  0.655  0.763  0.765  0.750
XLM-R (Meaning)  0.695  0.715  0.723  0.653  0.746  0.748  0.742
Avg              0.708  0.728  0.726  0.654  0.755  0.756  0.746


In [15]:
df_results.loc['XLM-R'].mean()

0.7321005389052685

In [16]:
df_results.loc['XLM-R (Meaning)'].mean()

0.7174894644738702